# Training Passive Liveness Detection Model
Notebook ini dibuat untuk melatih model mendeteksi Liveness (Live vs Spoof).

In [1]:
import os
import cv2
import numpy as np
import gc
from mtcnn import MTCNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
import pickle

detector = MTCNN()

In [2]:
def apply_clahe_and_resize(face, size=(128, 128)):
    lab = cv2.cvtColor(face, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    lab = cv2.merge((l,a,b))
    face = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    face = cv2.resize(face, size, interpolation=cv2.INTER_AREA)
    return face.astype("float32")

def detect_and_preprocess_face(img):
    h, w = img.shape[:2]
    scale = 640 / max(h, w)
    if scale < 1:
        img = cv2.resize(img, (int(w * scale), int(h * scale)))
        
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = detector.detect_faces(rgb)
    results = [r for r in results if r['confidence'] >= 0.8]
    
    if len(results) > 0:
        results = sorted(results, key=lambda x: x['box'][2] * x['box'][3], reverse=True)
        result = results[0]
        x, y, w, h = result['box']
        pad = int(min(w, h) * 0.1)
        x1, y1 = max(0, x-pad), max(0, y-pad)
        x2, y2 = min(rgb.shape[1], x+w+pad), min(rgb.shape[0], y+h+pad)
        face = rgb[y1:y2, x1:x2]
    else:
        # Jika wajah tidak terdeteksi, asumsikan gambar sudah pre-cropped
        face = rgb
        
    if face.size == 0:
        return None
        
    return apply_clahe_and_resize(face)

In [3]:
def load_dataset(base_folder):
    images = []
    labels = []
    skipped = 0
    classes = ['live', 'spoof']
    
    print(f"[INFO] Loading from {base_folder}...")
    for class_name in classes:
        folder_path = os.path.join(base_folder, class_name)
        if not os.path.isdir(folder_path):
            continue
            
        files = os.listdir(folder_path)
        used = 0
        for i, name in enumerate(files):
            if not name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
                
            img_path = os.path.join(folder_path, name)
            img = cv2.imread(img_path)
            if img is None:
                skipped += 1
                continue
                
            try:
                processed_img = detect_and_preprocess_face(img)
            except Exception as e:
                skipped += 1
                continue
                
            if processed_img is not None:
                images.append(processed_img)
                labels.append(class_name)
                used += 1
            else:
                skipped += 1
                
            if i % 50 == 0:
                print(f"Processed {i}/{len(files)} for {class_name}", end="\r")
                gc.collect()
                
        print(f"\n[OK] {class_name}: {used} images loaded.")
        
    return np.array(images, dtype="float32"), np.array(labels)

# Memuat data train dan test
x_train, y_train_labels = load_dataset('Dataset/Dataset_liveness/train')
x_test, y_test_labels = load_dataset('Dataset/Dataset_liveness/test')

print("x_train shape :", x_train.shape)
print("x_test shape  :", x_test.shape)

[INFO] Loading from Dataset/Dataset_liveness/train...
Processed 2050/2055 for live
[OK] live: 2055 images loaded.
Processed 2500/2525 for spoof
[OK] spoof: 2525 images loaded.
[INFO] Loading from Dataset/Dataset_liveness/test...
Processed 500/514 for live
[OK] live: 514 images loaded.
Processed 600/631 for spoof
[OK] spoof: 631 images loaded.
x_train shape : (4580, 128, 128, 3)
x_test shape  : (1145, 128, 128, 3)


In [ ]:
# ============================================================
# VISUALISASI DISTRIBUSI DATASET (Live vs Spoof)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

classes      = ['live', 'spoof']
train_counts = [int(np.sum(y_train_labels == c)) for c in classes]
test_counts  = [int(np.sum(y_test_labels  == c)) for c in classes]
total_counts = [t + v for t, v in zip(train_counts, test_counts)]

x     = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 6))

bars_train = ax.bar(x - width, train_counts, width, label='Train',
                    color=['#4CAF50', '#F44336'], edgecolor='black', linewidth=0.8)
bars_test  = ax.bar(x,         test_counts,  width, label='Test',
                    color=['#81C784', '#E57373'], edgecolor='black', linewidth=0.8)
bars_total = ax.bar(x + width, total_counts, width, label='Total',
                    color=['#1B5E20', '#B71C1C'], edgecolor='black', linewidth=0.8)

for bar in list(bars_train) + list(bars_test) + list(bars_total):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., h + 15,
            f'{int(h)}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Distribusi Dataset Liveness Detection (Live vs Spoof)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Kelas', fontsize=12)
ax.set_ylabel('Jumlah Gambar', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in classes], fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(0, max(total_counts) * 1.18)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabel ringkasan
print('=' * 48)
print(f"{'Kelas':<10} {'Train':>8} {'Test':>8} {'Total':>8}")
print('-' * 48)
for c, tr, te, tot in zip(classes, train_counts, test_counts, total_counts):
    print(f"{c.capitalize():<10} {tr:>8} {te:>8} {tot:>8}")
print('-' * 48)
print(f"{'TOTAL':<10} {sum(train_counts):>8} {sum(test_counts):>8} {sum(total_counts):>8}")
print('=' * 48)


In [5]:
# Normalisasi data
x_train = x_train / 255.0
x_test = x_test / 255.0

# Encoding label: live -> 0, spoof -> 1
le = LabelEncoder()
y_train = le.fit_transform(y_train_labels)
y_test = le.transform(y_test_labels)

# Simpan label encoder\
with open("Label_encoder_liveness.pickle", "wb") as f:
    pickle.dump(le, f)

print("Classes:", le.classes_)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Classes: ['live' 'spoof']
y_train shape: (4580,)
y_test shape: (1145,)


In [6]:
# Model CNN untuk Liveness (Binary Classification)
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Sigmoid untuk binary classification (live/spoof)
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

d:\ABSENSI_VC\venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,665 (12.61 MB)

 Trainable params: 3,305,217 (12.61 MB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
# Augmentasi data (hanya untuk data training)
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Callbacks
checkpoint = ModelCheckpoint('liveness_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Training
epochs = 30
batch_size = 32

history = model.fit(
    datagen.flow(x_train, y_train, batch_size=batch_size),
    validation_data=(x_test, y_test),
    epochs=epochs,
    callbacks=[checkpoint, early_stop]
)

Epoch 1/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 633ms/step - accuracy: 0.6371 - loss: 1.4166
Epoch 1: val_accuracy improved from None to 0.46288, saving model to liveness_model.keras
144/144 ━━━━━━━━━━━━━━━━━━━━ 101s 668ms/step - accuracy: 0.6843 - loss: 0.8828 - val_accuracy: 0.4629 - val_loss: 5.8682
Epoch 2/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 632ms/step - accuracy: 0.7682 - loss: 0.4936
Epoch 2: val_accuracy did not improve from 0.46288
144/144 ━━━━━━━━━━━━━━━━━━━━ 95s 662ms/step - accuracy: 0.7662 - loss: 0.5043 - val_accuracy: 0.4629 - val_loss: 1.5592
Epoch 3/30
 46/144 ━━━━━━━━━━━━━━━━━━━━ 1:05 665ms/step - accuracy: 0.7831 - loss: 0.4789

### Pengujian Model secara Real-Time
Jalankan cell di bawah ini untuk membuka webcam dan menguji model yang baru saja dilatih.

In [ ]:
import cv2
import numpy as np

print("[INFO] Membuka Webcam...")
cap = cv2.VideoCapture(0)

print("Tekan 'q' pada keyboard di jendela kamera untuk keluar.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = detector.detect_faces(rgb_frame)
    
    for result in results:
        if result['confidence'] < 0.8:
            continue
            
        x, y, w, h = result['box']
        pad = int(min(w, h) * 0.1)
        x1, y1 = max(0, x-pad), max(0, y-pad)
        x2, y2 = min(frame.shape[1], x+w+pad), min(frame.shape[0], y+h+pad)
        
        face_crop = rgb_frame[y1:y2, x1:x2]
        if face_crop.size == 0:
            continue
            
        # Preprocessing 
        processed_face = apply_clahe_and_resize(face_crop)
        processed_face = processed_face / 255.0  # Normalisasi
        processed_face = np.expand_dims(processed_face, axis=0)
        
        # Prediksi
        pred = model.predict(processed_face, verbose=0)[0][0]
        
        if pred < 0.5:
            label = "Live"
            color = (0, 255, 0)
            confidence = (1 - pred) * 100
        else:
            label = "Spoof"
            color = (0, 0, 255)
            confidence = pred * 100
            
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        text = f"{label}: {confidence:.1f}%"
        (text_width, text_height), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (x1, y1 - 25), (x1 + text_width, y1), color, -1)
        cv2.putText(frame, text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
    cv2.imshow('Liveness Test - Tekan Q untuk keluar', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


[INFO] Membuka Webcam...
Tekan 'q' pada keyboard di jendela kamera untuk keluar.
